# Exploración y preprocesamiento de datos

**Proyecto:** Predicción de la gravedad de siniestros viales en CABA

**Autor:** Juan Pablo Monllor - Aprendizaje Automático 2026

Esta notebook carga los datos originales, los explora, documenta las decisiones de limpieza y genera el dataset procesado que se usará para el modelado.

In [9]:
# Librerías para manejar datos (pandas, numpy) y hacer gráficos (matplotlib)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Carga de datos

In [10]:
# Cargo el dataset de siniestros viales de CABA (Buenos Aires Data).
# El archivo usa punto y coma como separador, por eso indico sep=";".
# low_memory=False: algunas columnas mezclan números con el texto "SD" (sin dato);
# así pandas lee el archivo completo de una vez y evita el aviso de tipos mezclados.
siniestros = pd.read_csv("../data/raw/siniestros_viales_hechos.csv", sep=";", low_memory=False)

print("Filas y columnas:", siniestros.shape)
siniestros.head()

Filas y columnas: (65818, 21)


,id_siniestro,numero_total_de_victimas,numero_victimas_leve_siniestro,numero_victimas_grave_siniestro,numero_victimas_mortal_siniestro,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,hora_siniestro,...,direccion_normalizada_siniestro,comuna_siniestro,tipo_de_via_siniestro,geocodificacion_plana,longitud_siniestro,latitud_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,gravedad_siniestro
0,LC-2019-0000053,1,1,0,0,2019-01-01,2019,1,1,01:55:00,...,NaN,Comuna 8,AUTOPISTA,POINT (101813.84712503915943671 95578.55507230...,-58.44351,-34.669125,AUTO-UTILITARIO,AUTO,UTILITARIO,LEVE
1,LC-2019-0000647,1,0,1,0,2019-01-01,2019,1,1,18:50:00,...,DE LOS CONSTITUYENTES AV. y HABANA,Comuna 12,AVENIDA,POINT (97510.27137648081406951 105087.83369881...,-58.490436,-34.583403,MOTO-AUTO,MOTO,AUTO,GRAVE
2,LC-2019-0000445,3,3,0,0,2019-01-01,2019,1,1,13:40:00,...,NaN,Comuna 11,NaN,POINT (94371.44237885093025398 102884.19365105...,-58.524662,-34.603255,AUTO-AUTO,AUTO,AUTO,LEVE
3,LC-2019-0000194,1,1,0,0,2019-01-01,2019,1,1,07:15:00,...,NaN,Comuna 9,NaN,POINT (94030.76669932194636203 97681.071761248...,-58.528413,-34.650156,AUTO-CAMION,AUTO,CAMION,LEVE
4,LC-2019-0000329,1,1,0,0,2019-01-01,2019,1,1,12:24:00,...,NaN,Comuna 4,NaN,POINT (105367.86665769023238681 97877.75085908...,-58.404748,-34.648387,AUTO-MOVIL,AUTO,MOVIL,LEVE


In [11]:
# Cargo la lluvia por hora de Open-Meteo.
# Las primeras 3 líneas del archivo no son datos: describen el punto
# (latitud, longitud, altura y zona horaria). Por eso las salteo con skiprows=3.
lluvia = pd.read_csv("../data/external/lluvia_caba_open_meteo_2019_2025.csv", skiprows=3)

print("Filas y columnas:", lluvia.shape)
lluvia.head()

Filas y columnas: (61368, 2)


,time,precipitation (mm)
0,2019-01-01T00:00,0.0
1,2019-01-01T01:00,0.0
2,2019-01-01T02:00,0.0
3,2019-01-01T03:00,0.0
4,2019-01-01T04:00,0.0


## 2. Primera inspección

In [12]:
# info() muestra cada columna con su tipo de dato y cuántos valores no nulos tiene.
# Sirve para saber qué columnas son numéricas, cuáles son texto y dónde hay faltantes.
siniestros.info()

<class 'pandas.DataFrame'>
RangeIndex: 65818 entries, 0 to 65817
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   id_siniestro                      65818 non-null  str  
 1   numero_total_de_victimas          65818 non-null  str  
 2   numero_victimas_leve_siniestro    65818 non-null  int64
 3   numero_victimas_grave_siniestro   65818 non-null  int64
 4   numero_victimas_mortal_siniestro  65818 non-null  int64
 5   fecha_siniestro                   65818 non-null  str  
 6   anio_siniestro                    65818 non-null  int64
 7   mes_siniestro                     65818 non-null  int64
 8   dia_siniestro                     65818 non-null  int64
 9   hora_siniestro                    65741 non-null  str  
 10  rango_horario                     65741 non-null  str  
 11  direccion_normalizada_siniestro   52918 non-null  str  
 12  comuna_siniestro                  62801 non

**Observaciones:** el dataset tiene 65.818 siniestros y 21 columnas (6 numéricas y 15 de texto). La columna `numero_total_de_victimas` aparece como texto porque mezcla números con "SD". Entre las variables que se van a usar como predictoras, `tipo_de_via_siniestro` tiene 12.227 nulos (19%), `comuna_siniestro` 3.017 (5%) y `rango_horario` 77 (0,1%). La decisión sobre estos faltantes se toma en la sección 5.

In [14]:
# Cuento los valores faltantes de dos formas:
# - nulos: celdas vacías (NaN)
# - "SD": el dataset usa el texto "SD" (sin dato) cuando no se registró la información
faltantes = pd.DataFrame({
    "nulos": siniestros.isna().sum(),
    "SD": (siniestros == "SD").sum()
})

# Sumo nulos y "SD" (los dos significan "sin dato") y calculo el porcentaje
# sobre el total de filas del dataset
faltantes["total"] = faltantes["nulos"] + faltantes["SD"]
faltantes["porcentaje"] = (faltantes["total"] / len(siniestros) * 100).round(1)

# Muestro solo las columnas que tienen algún faltante, de mayor a menor
faltantes[faltantes["total"] > 0].sort_values("porcentaje", ascending=False)

,nulos,SD,total,porcentaje
contraparte_siniestro,0,14066,14066,21.4
direccion_normalizada_siniestro,12900,727,13627,20.7
tipo_de_via_siniestro,12227,315,12542,19.1
modo_desplazamiento_victima,0,10928,10928,16.6
comuna_siniestro,3017,535,3552,5.4
numero_total_de_victimas,0,3276,3276,5.0
longitud_siniestro,2378,359,2737,4.2
latitud_siniestro,2378,359,2737,4.2
geocodificacion_plana,2367,7,2374,3.6
hora_siniestro,77,4,81,0.1


**Observaciones:** el dataset registra los datos faltantes de dos formas: celdas vacías (nulos) y el texto "SD" (sin dato). Sumando ambas, las predictoras con más faltantes son la contraparte (21,4%), el tipo de vía (19,1%) y el modo de desplazamiento de la víctima (16,6%); la comuna tiene un 5,4% y el rango horario un 0,1%. Eliminar todas las filas con algún faltante implicaría perder una parte importante de los datos, por lo que su tratamiento se analiza en la sección 5.

In [15]:
# Reviso si hay filas repetidas y si algún siniestro aparece más de una vez.
# Cada fila debería ser un siniestro distinto.
print("Filas duplicadas:", siniestros.duplicated().sum())
print("id_siniestro repetidos:", siniestros["id_siniestro"].duplicated().sum())

Filas duplicadas: 0
id_siniestro repetidos: 0


**Observaciones:** no hay filas duplicadas ni siniestros repetidos: cada fila corresponde a un siniestro distinto (`id_siniestro` único).

In [16]:
# Reviso tipos de dato y nulos en el archivo de lluvia.
lluvia.info()
print()
print("Nulos por columna:")
print(lluvia.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 61368 entries, 0 to 61367
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   time                61368 non-null  str    
 1   precipitation (mm)  61368 non-null  float64
dtypes: float64(1), str(1)
memory usage: 1.9 MB

Nulos por columna:
time                  0
precipitation (mm)    0
dtype: int64


**Observaciones:** el archivo de lluvia tiene 61.368 registros horarios (2019-2025) y no tiene valores nulos. La columna `time` está como texto: se convertirá a fecha y hora en la sección 6, para unirla con los siniestros.